# GoLLIE-13B 通用因果关系抽取 baseline

本 notebook 使用 Sainz et al. (ICLR 2024) 的 **GoLLIE（Guideline following Large Language Model for Information Extraction）**，在本项目四个正式数据入口上执行单阶段联合检测与抽取，并交给同一个 `src.evaluator.Evaluator`。

固定实现边界：

- 模型：`HiTZ/GoLLIE-13B` 的第三方 `Q8_0 GGUF` 转换，通过 LM Studio 本地运行；论文中应写作 **GoLLIE-13B (Q8_0 GGUF), executed locally through LM Studio**。
- 输入：作者 Relation Extraction 模板，即 Python `@dataclass` guideline、原始 `text` 与 `result = [`；不使用 chat template、few-shot、RAG、gold 标签或 gold span。
- 解码：raw completion、greedy (`temperature=0`)、无 thinking/CoT；列表为空即 detection negative，列表非空即 positive。
- schema：四个数据集共用同一份 `CausalRelation` guideline；`arg1` 映射 cause，`arg2` 映射 effect；保留所有模型预测，不做语义修补。
- 评估：同时报告 detection、Anchor all-samples extraction 与 detected-only 诊断；正式主比较使用 all-samples。

In [ ]:
from pathlib import Path
import logging
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "evaluator.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("找不到 Master thesis 项目根目录；请从项目内启动 notebook。")
if Path(sys.prefix).name.lower() != "master_thesis":
    raise RuntimeError(f"当前解释器为 {sys.executable}，请切换到 Master_thesis kernel。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("gollie_baseline_notebook")

SOURCE_DIR = ROOT / "reference code from related work" / "GoLLIE-main" / "GoLLIE-main"
LMSTUDIO_BASE_URL = "http://127.0.0.1:1234/v1"
MODEL_ID = "hitz-gollie-13b-assafetensors"
DATASET_NAMES = ["cnc_sft_test", "li", "ade", "politicause"]
PRIMARY_METRIC = "anchor_window"
RUN_ID = "gollie13b_q8_0_author_relation_template_v1"
OUTPUT_DIR = ROOT / "results" / "eval_report" / "gollie_baseline"
REUSE_CHECKPOINT = True

RUN_SMOKE = True
EVAL_SAMPLE_N = 10
RUN_FULL_EVAL = False
FULL_EVAL_SAMPLE_N = None

logger.info("项目：%s；解释器：%s；模型：%s", ROOT, sys.executable, MODEL_ID)

## 1. 环境和模型核对

模型由 LM Studio 提供服务，所以继续使用项目现有 `Master_thesis` 环境，不安装 GoLLIE 原仓库的 Transformers、FlashAttention 或 bitsandbytes 依赖，也不会影响其他 notebook。正式运行前会从 LM Studio 原生接口核对模型确实为 13B、GGUF、Q8_0、已加载，并把实际加载配置写入 manifest。

In [ ]:
import pandas as pd
from IPython.display import HTML, Markdown, display
from src.data_io import DATASET_FILES, load_dataset
from src.evaluator import primary_metric_for_dataset
from src.gollie_baseline import (
    LMStudioGoLLIERunner, RunConfig, build_prompt, run_gollie_baseline,
    software_versions, source_hashes,
)

CONFIG = RunConfig(
    model=MODEL_ID,
    base_url=LMSTUDIO_BASE_URL,
    max_tokens=1024,
    temperature=0.0,
    seed=4000,
    cache_prompt=False,
)
AUTHOR_SOURCE_HASHES = source_hashes(SOURCE_DIR)
display(pd.DataFrame([{"Package": name, "Version": version or "NOT INSTALLED"}
                      for name, version in software_versions().items()]))
display(pd.DataFrame([{
    "Model": MODEL_ID,
    "Backend": "LM Studio raw /v1/completions",
    "Quantization": CONFIG.quantization,
    "Decoding": "greedy; no thinking; no chat template",
    "Max output tokens": CONFIG.max_tokens,
    "Prompt cache": CONFIG.cache_prompt,
}]))


def require_inference_environment() -> dict:
    """运行前核对作者源码及 LM Studio 中实际加载的 Q8_0 模型。"""
    runner = LMStudioGoLLIERunner(CONFIG, SOURCE_DIR)
    runner.require_model()
    metadata = runner.model_metadata()
    logger.info("LM Studio 已加载：%s；量化：%s；context=%s",
                metadata["display_name"], metadata["quantization"]["name"],
                metadata["loaded_instances"][0]["config"].get("context_length"))
    return metadata


## 2. 数据和固定 guideline

四个数据集均直接读取项目清洗后的正式入口。下表只用于核对样本规模；gold 不进入 prompt。Prompt 预览应以 `result = [` 结束，这是作者 README 和 Relation Extraction 模板规定的生成起点。

In [ ]:
if EVAL_SAMPLE_N < 1:
    raise ValueError("EVAL_SAMPLE_N 必须为正整数。")
if FULL_EVAL_SAMPLE_N is not None and FULL_EVAL_SAMPLE_N < 1:
    raise ValueError("FULL_EVAL_SAMPLE_N 必须为 None 或正整数。")

DATASETS = {name: load_dataset(name) for name in DATASET_NAMES}
rows = []
for name, samples in DATASETS.items():
    rows.append({
        "Dataset": name,
        "Input": DATASET_FILES[name],
        "N": len(samples),
        "Positive": sum(bool(sample["has_causal"]) for sample in samples),
        "Gold pairs": sum(len(sample.get("relations", [])) for sample in samples),
        "Max pairs/sample": max(len(sample.get("relations", [])) for sample in samples),
        "Primary extraction metric": PRIMARY_METRIC or primary_metric_for_dataset(name),
    })
display(pd.DataFrame(rows))
preview = build_prompt(DATASETS[DATASET_NAMES[0]][0])
display(Markdown("### GoLLIE 实际输入预览\n```python\n" + preview + "\n```"))
assert DATASETS[DATASET_NAMES[0]][0]["text"] in preview


## 3. Smoke test

默认对每个数据集前 10 条运行一次工程 smoke。每条请求完成后立即追加到 raw JSONL；若 kernel、LM Studio 或机器中断，下次从同一输入的已完成连续前缀续跑。进度条按数据集分别显示。

In [ ]:
def result_row(result: dict) -> dict:
    """提取论文表格需要的端到端结果和解析诊断。"""
    report = result["report"]
    metric = report["extraction"]["primary_metric"]
    extraction = report["extraction"][metric]
    diagnostics = report["baseline"]["diagnostics"]
    return {
        "Dataset": result["dataset"],
        "N": result["n_samples"],
        "Detection F1": report["detection"]["f1"],
        "Extraction metric": metric,
        "Extraction F1 (all)": extraction["all_samples"]["f1"],
        "Extraction F1 (detected-only)": extraction["detected_only"]["f1"],
        "Parse-error samples": diagnostics["parse_error_samples"],
        "Nonverbatim pairs": diagnostics["nonverbatim_pairs"],
        "Report": str(result["paths"]["report_md"]),
    }


def make_progress(phase: str, dataset: str):
    """创建 notebook 内可更新的单数据集进度条。"""
    handle = None

    def update(completed: int, total: int) -> None:
        nonlocal handle
        percent = 100.0 if total == 0 else completed / total * 100.0
        widget = HTML(
            f"<div style='margin:6px 0 10px 0'>"
            f"<div><strong>{phase} / {dataset}</strong>: {completed}/{total} ({percent:.1f}%)</div>"
            f"<progress value='{percent:.4f}' max='100' style='width:100%;height:18px'></progress></div>"
        )
        if handle is None:
            handle = display(widget, display_id=True)
        elif handle is not None:
            handle.update(widget)

    return update


def run_selected_datasets(phase: str, sample_n: int | None) -> list[dict]:
    """依次运行四个数据集，并用统一 evaluator 立即保存和展示结果。"""
    metadata = require_inference_environment()
    display(pd.DataFrame([{
        "Loaded model": metadata["display_name"],
        "Format": metadata["format"],
        "Quantization": metadata["quantization"]["name"],
        "Size (GB)": round(metadata["size_bytes"] / 1e9, 2),
        "Context": metadata["loaded_instances"][0]["config"].get("context_length"),
        "Flash attention": metadata["loaded_instances"][0]["config"].get("flash_attention"),
    }]))
    results = []
    for dataset in DATASET_NAMES:
        samples = DATASETS[dataset] if sample_n is None else DATASETS[dataset][:sample_n]
        runner = LMStudioGoLLIERunner(CONFIG, SOURCE_DIR)
        runner.require_model()
        result = run_gollie_baseline(
            samples=samples,
            runner=runner,
            dataset=dataset,
            output_dir=OUTPUT_DIR / phase / dataset,
            run_name=f"{RUN_ID}_{dataset}_n{len(samples)}",
            primary_metric=PRIMARY_METRIC,
            reuse_checkpoint=REUSE_CHECKPOINT,
            progress_callback=make_progress(phase, dataset),
        )
        results.append(result)
        fence = chr(96) * 3
        display(Markdown(fence + "text\n" + result["formatted_report"] + "\n" + fence))
    return results


smoke_results = []
if RUN_SMOKE:
    smoke_results = run_selected_datasets("smoke", EVAL_SAMPLE_N)
    display(pd.DataFrame([result_row(result) for result in smoke_results]))
else:
    logger.info("Smoke 已关闭；未调用 LM Studio。")

## 4. 全量正式运行

确认 smoke 的接口、列表解析和非原文 span 数量后，将 `RUN_FULL_EVAL=True`、`FULL_EVAL_SAMPLE_N=None`，再执行本节。全量与 smoke 使用独立目录；正式主结果是 detection 与 **Extraction F1 (all)**，detected-only 只解释抽取器在检测成功子集上的表现。

In [ ]:
full_results = []
if RUN_FULL_EVAL:
    full_results = run_selected_datasets("full", FULL_EVAL_SAMPLE_N)
    display(pd.DataFrame([result_row(result) for result in full_results]))
else:
    logger.info("全量评估已关闭。开启方式：RUN_FULL_EVAL=True，FULL_EVAL_SAMPLE_N=None。")

## 5. 输出与复核

每个数据集目录含无 gold 的输入 CSV、原始 completion、evaluator predictions、JSON/Markdown 报告及 manifest。Manifest 固定记录作者源码哈希、prompt 哈希、实际模型 ID、Q8_0、LM Studio 加载参数、raw completion、greedy 和无 thinking。

In [ ]:
import json

recent_results = full_results or smoke_results
if recent_results:
    first = recent_results[0]
    display(pd.DataFrame([{"File": name, "Path": str(path)}
                          for name, path in first["paths"].items()]))
    with first["paths"]["raw"].open(encoding="utf-8") as file:
        raw_rows = [json.loads(line) for _, line in zip(range(5), file)]
    display(Markdown("### 原始 GoLLIE completion 前 5 条"))
    display(pd.DataFrame(raw_rows))
else:
    logger.info("尚未运行模型。预期输出目录：%s", OUTPUT_DIR)